<a href="https://colab.research.google.com/github/dikshanaa-m/minutes-of-the-meeting-generator/blob/main/mom_generator_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install nltk spacy scikit-learn textblob -q
!python -m spacy download en_core_web_sm -q

import nltk, re, spacy, numpy as np
nltk.download('punkt',                        quiet=True)
nltk.download('punkt_tab',                    quiet=True)
nltk.download('stopwords',                    quiet=True)
nltk.download('averaged_perceptron_tagger',   quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk import pos_tag, ngrams
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from textblob import TextBlob

nlp        = spacy.load("en_core_web_sm")
stop_words = set(stopwords.words('english'))
stemmer    = PorterStemmer()

#
print("📝 Paste your meeting transcript. Type END on a new line when done.\n")
lines = []
while True:
    line = input()
    if line.strip().upper() == "END":
        break
    if line.strip() == "" and not user_lines if 'user_lines' in dir() else False:
        break
    lines.append(line)

transcript = "\n".join(lines)

if not transcript.strip():
    print("⚠️  No input. Using sample transcript.\n")
    transcript = """
    Meeting date: March 15, 2024. Attendees: Alice Johnson, Bob Smith, Carol Lee, David Kumar.
    Alice opened the meeting by discussing the Q1 marketing budget. She mentioned that
    the budget needs to be finalized by March 20, 2024. Bob Smith raised concerns about
    the social media campaign costs being too high. Carol agreed with Bob and suggested
    reducing Instagram ads by 30 percent.
    David Kumar from the engineering team said the new product feature will be ready
    by April 1. Alice asked David to send a demo link to the team by next Monday.
    Bob will prepare a revised budget proposal for review. Carol Lee will coordinate
    with the design team at Creative Studio to update the brand guidelines.
    The next meeting is scheduled for March 22 at 10 AM.
    Alice concluded that all action items must be tracked in Jira.
    The team agreed to use Slack for daily updates going forward.
    """

print(f"✅ Processing {len(transcript.split())} words...\n")


# NLP STEP 1 — TOKENIZATION
sentences = sent_tokenize(transcript)
words     = word_tokenize(transcript.lower())

# NLP STEP 2 — STOPWORD REMOVAL + LOWERCASING

cleaned = [w for w in words if w.isalnum() and w not in stop_words]

# NLP STEP 3 — STEMMING

stemmed = [stemmer.stem(w) for w in cleaned]

# NLP STEP 4 — POS TAGGING
pos_tags = pos_tag(word_tokenize(transcript))
nouns    = [w for w, tag in pos_tags if tag in ('NN','NNS','NNP','NNPS')]
verbs    = [w for w, tag in pos_tags if tag in ('VB','VBZ','VBD','VBG')]

# NLP STEP 5 — TF-IDF SCORING

vectorizer     = TfidfVectorizer(stop_words='english')
tfidf_matrix   = vectorizer.fit_transform(sentences)
sent_scores    = np.array(tfidf_matrix.sum(axis=1)).flatten()
top_n          = min(4, len(sentences))
top_sentences  = [sentences[i] for i in sent_scores.argsort()[-top_n:][::-1]]
feature_names  = vectorizer.get_feature_names_out()
word_scores    = np.array(tfidf_matrix.sum(axis=0)).flatten()
keywords       = [feature_names[i] for i in word_scores.argsort()[-8:][::-1]]


# NLP STEP 6 — NAMED ENTITY RECOGNITION (NER)

doc      = nlp(transcript)
entities = {"PERSON": [], "ORG": [], "DATE": [], "TIME": []}
for ent in doc.ents:
    if ent.label_ in entities and ent.text.strip() not in entities[ent.label_]:
        entities[ent.label_].append(ent.text.strip())


# NLP STEP 8 — N-GRAM EXTRACTION

bigrams  = list(ngrams(cleaned, 2))
trigrams = list(ngrams(cleaned, 3))
top_bigrams  = Counter(bigrams).most_common(5)
top_trigrams = Counter(trigrams).most_common(3)

# NLP STEP 9 — RULE-BASED CLASSIFICATION

action_pattern   = r'\b(will|must|should|needs? to|agreed to|asked to|prepare|send|coordinate|finalize|assigned to)\b'
decision_pattern = r'\b(decided|agreed|resolved|confirmed|finalized|concluded)\b'

action_items = []
decisions    = []
key_points   = []

for s in sentences:
    if re.search(decision_pattern, s, re.IGNORECASE):
        decisions.append(s.strip())
    elif re.search(action_pattern, s, re.IGNORECASE):
        action_items.append(s.strip())
    elif s in top_sentences:
        key_points.append(s.strip())
═══════
# OUTPUT — GENERATED MEETING MINUTES

print("=" * 65)
print("           📋  MEETING MINUTES  (Auto-Generated)")
print("=" * 65)

print("\n📅  DATE :", ", ".join(entities["DATE"]) or "N/A")
print("\n👥  ATTENDEES:")
for p in entities["PERSON"]:  print(f"      • {p}")
print("\n🏢  ORGANISATIONS:")
for o in entities["ORG"]:     print(f"      • {o}")

print("\n🔑 (TF-IDF):", ", ".join(keywords[:6]))

print("\n💬  TOP PHRASES (N-grams):")
for bg, count in top_bigrams:
    print(f"      • {' '.join(bg)} ({count}x)")

print("\n📝  KEY DISCUSSION POINTS:")
for s in key_points[:3]:      print(f"      → {s[:110]}")

print("\n✅  DECISIONS MADE:")
for i,d in enumerate(decisions[:4],1):   print(f"      {i}. {d[:110]}")

print("\n📌  ACTION ITEMS:")
for i,a in enumerate(action_items[:5],1): print(f"      {i}. {a[:110]}")



print("\n🏷️   POS INSIGHT:")
print(f"      Top nouns : {', '.join(list(dict.fromkeys(nouns))[:6])}")
print(f"      Top verbs : {', '.join(list(dict.fromkeys(verbs))[:6])}")

print("\n" + "=" * 65)
print("  NLP Pipeline: Tokenization | Stopwords | Stemming | POS Tagging")
print("              | TF-IDF | NER | Sentiment | N-grams | Rule-Based")
print("=" * 65)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 47.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
📝 Paste your meeting transcript. Type END on a new line when done.

Meeting date: April 10, 2024. Attendees: Karan Patel, Meera Nair, Suresh Babu, Divya Menon, Rohan Verma.  Karan Patel opened the meeting and said the main agenda for today is to review  the college fest planning progress and assign remaining responsibilities.  Meera Nair from the events team said the venue booking for the main auditorium  has been confirmed. She mentioned that the decoration committee has already  started working on the theme. Meera said the theme for this year is Neon Nights  and the decorati